<a href="https://colab.research.google.com/github/arushikumarics24-beep/BIS_LAB/blob/main/Parallel_cellular_algorithm_week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np
from concurrent.futures import ThreadPoolExecutor

# Generator parameters
Pmin = np.array([50, 50, 50])
Pmax = np.array([200, 250, 300])

a = np.array([0.003, 0.002, 0.0015])
b = np.array([5.3, 5.5, 5.8])
c = np.array([500, 400, 200])

demand = 450

# Algorithm parameters
ROWS = 5
COLS = 5
ITERATIONS = 100
PROCESSORS = 4
LEARNING_RATE = 0.5
MUTATION_RATE = 0.2
MUTATION_SCALE = 10


# Repair solution to satisfy demand and capacity
def repair(P):
    P = np.clip(P, Pmin, Pmax)

    diff = demand - np.sum(P)

    while abs(diff) > 1e-8:
        if diff > 0:
            available = Pmax - P
        else:
            available = P - Pmin

        indices = np.where(available > 1e-8)[0]

        if len(indices) == 0:
            break

        change = diff / len(indices)

        for i in indices:
            amount = min(abs(change), available[i])
            P[i] += amount if diff > 0 else -amount

        diff = demand - np.sum(P)

    return P


# Calculate generation cost
def fitness(P):
    return np.sum(a * P**2 + b * P + c)


# Initialize cellular grid
grid = np.array([
    [repair(np.random.uniform(Pmin, Pmax))
     for _ in range(COLS)]
    for _ in range(ROWS)
])


# Update one cell using its neighborhood
def update_cell(args):
    i, j, old_grid, seed = args
    rng = np.random.default_rng(seed)

    neighbors = []

    for x in range(max(0, i-1), min(ROWS, i+2)):
        for y in range(max(0, j-1), min(COLS, j+2)):
            neighbors.append(old_grid[x, y])

    best_neighbor = min(neighbors, key=fitness)

    current = old_grid[i, j].copy()

    new = current + LEARNING_RATE * (
        best_neighbor - current
    )

    if rng.random() < MUTATION_RATE:
        new += rng.normal(0, MUTATION_SCALE, size=3)

    return repair(new)


# Parallel Cellular Algorithm
for iteration in range(ITERATIONS):

    old_grid = grid.copy()

    tasks = [
        (i, j, old_grid, iteration * ROWS * COLS + i * COLS + j)
        for i in range(ROWS)
        for j in range(COLS)
    ]

    # Parallel update of cells
    with ThreadPoolExecutor(max_workers=PROCESSORS) as executor:
        results = list(executor.map(update_cell, tasks))

    # Synchronize and update grid simultaneously
    grid = np.array(results).reshape(ROWS, COLS, 3)

# Find best solution
best = min(grid.reshape(-1, 3), key=fitness)

print("Parallel Cellular Algorithm")
print("Generator Outputs:", np.round(best, 2))
print("Total Power:", round(np.sum(best), 2), "MW")
print("Minimum Generation Cost:", round(fitness(best), 2))


Parallel Cellular Algorithm
Generator Outputs: [148.14 172.23 129.63]
Total Power: 450.0 MW
Minimum Generation Cost: 3734.63
